# 🏠 House Price Prediction: My First End-to-End ML Project

> *"I built a model to predict house prices. The model politely informed me that the data had other plans."* 😄

---

## 📌 Project at a glance

| | |
|---|---|
| **Goal** | Predict the price of a house from its features (area, bedrooms, location, etc.) |
| **Type of problem** | Supervised Learning → **Regression** |
| **Algorithm** | Linear Regression |
| **Metric** | R² Score |
| **Tools** | Python · Pandas · NumPy · Matplotlib · Seaborn · Scikit-learn |

## 🗺️ Roadmap of this notebook

1. 📥 Load the data
2. 🧹 Data cleaning & sanity checks
3. 🔍 Univariate analysis (one column at a time)
4. 🔗 Bivariate analysis (each feature vs. Price)
5. 🔥 Correlation matrix
6. 🔤 Encoding categorical features
7. ✂️ Train-test split
8. 🤖 Linear Regression model
9. 📏 Evaluation
10. 🧠 Key findings & lessons learned

---

## 1️⃣ Importing libraries
`numpy` and `pandas` handle numbers and tables. The rest of this cell is the default Kaggle starter code that lists the files available in the environment.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## 2️⃣ Loading the dataset
We read the CSV file into a **DataFrame** (`df`), which is basically an Excel sheet that Python can work with.

In [ ]:
df=pd.read_csv('/kaggle/input/datasets/zafarali27/house-price-prediction-dataset/House Price Prediction Dataset.csv')

### 👀 First look at the data
`df.head()` shows the first 5 rows. It's the quickest way to understand what each column contains.

In [ ]:
df.head()

## 3️⃣ Data cleaning & sanity checks 🧹
Before any analysis, we check that the data is trustworthy:

- `describe()` → summary statistics (mean, min, max, etc.)
- `duplicated().sum()` → are there any repeated rows?
- `info()` → data types and missing values

In [ ]:
# data cleaning
df.describe()

In [ ]:
#No duplicates 
df.duplicated().sum()

In [ ]:
#No null values
df.info()

### 🔧 Fixing a data type
`Area` is converted to a float so numeric operations behave consistently.

In [ ]:
df['Area']=df['Area'].astype('Float64')

### 🆔 Checking the `Id` column
Every `Id` should appear exactly once. It's just a row label, so it carries **no information about price** and we'll drop it before modelling.

In [ ]:
df['Id'].value_counts()

In [ ]:
df['Area'].dtype

### ✅ Cleaning summary
- No duplicate rows
- No missing values
- Data types are correct
- Price is in dollars 💵

The data is clean, so we can move on to **Exploratory Data Analysis (EDA)**.

---
## 4️⃣ Exploratory Data Analysis (EDA) 🔍
### 4.1 Univariate analysis: numerical columns
We look at each numeric column on its own, using three tools:
- **Histogram** → shape of the distribution
- **Skewness** → how lopsided it is (close to 0 means symmetric)
- **Box plot** → outliers

#### 📐 Area

In [ ]:
#price is in dollors
#lets move to edA

In [ ]:
df.head()

In [ ]:
df['Area'].plot(kind='hist',bins=60)#kind of uniform can be other


**Reading the result:** `Area` has a skewness of about **-0.039**, which is nearly symmetric. The histogram is roughly uniform, meaning house sizes are spread evenly across the range.

In [ ]:
#The `Area` feature has a skewness of approximately -0.039, 
#indicating a nearly symmetric distribution with a negligible negative skew.

df['Area'].skew()

**Box plot:** no outliers in `Area`. ✅

#### 🏗️ YearBuilt

In [ ]:
df['Area'].plot(kind='box')#no outliers

In [ ]:
df['YearBuilt'].isnull().sum()#no null values

In [ ]:

df['YearBuilt'].plot(kind='hist',bins=60)

In [ ]:
df['YearBuilt'].skew()

In [ ]:
df['YearBuilt'].plot(kind='box')

#### 💰 Price (our target variable)
This is the column we want to predict. Its distribution matters a lot.

In [ ]:
df['Price'].plot(kind='hist', bins=40)#kind of uniform distribution

In [ ]:
df['Price'].skew()#It is nearly zero the negative skewness can be neglected and can assume the data is symmteric

In [ ]:
df['Price'].plot(kind='box')#from the box plot we can see there are no outliers prenet in the data

**Numerical columns, summary:**
- `Area`, `YearBuilt` and `Price` all have nearly zero skewness
- No outliers in any of them
- Distributions look roughly **uniform**, not bell-shaped

> 💡 Real-world prices are usually right-skewed with a long tail of expensive houses. A flat, uniform price distribution is the first hint that this dataset may not behave like real housing data.

---
### 4.2 Univariate analysis: categorical columns
For categories we use **bar charts** (counts) and **pie charts** (percentages).

**Result:**
- The frequency of each category is similar
- No single category is significantly more common than the others
- The dataset is relatively balanced

In [ ]:
#eda on categorical column
df['Location'].value_counts().plot(kind='bar')

In [ ]:
df['Location'].value_counts().plot(kind='pie',autopct='%0.01f%%')

In [ ]:
df['Bedrooms'].value_counts().plot(kind='bar')

In [ ]:
df['Bedrooms'].value_counts().plot(kind='pie',autopct='%0.01f%%')

In [ ]:
df['Bathrooms'].value_counts().plot(kind='bar')

In [ ]:
df['Bathrooms'].value_counts().plot(kind='pie',autopct='%0.01f%%')

In [ ]:
df['Floors'].value_counts().plot(kind='bar')

In [ ]:
df['Floors'].value_counts().plot(kind='pie',autopct='%0.01f%%')

**Garage:** 48% of houses have a garage and 52% don't, a nearly even split.

In [ ]:
#Garage = Yes (48%): Slightly more than half of the houses have a garage.
#Garage = No (52%): Nearly half of the houses do not have a garage.
df['Garage'].value_counts().plot(kind='bar')

In [ ]:
df['Garage'].value_counts().plot(kind='pie',autopct='%0.01f%%')

#### 🛠️ Condition

In [ ]:
df['Condition'].value_counts()

In [ ]:
df['Condition'].value_counts().plot(kind='bar')

In [ ]:
df['Condition'].value_counts().plot(kind='pie',autopct='%0.01f%%')

---
## 5️⃣ Bivariate analysis 🔗
This is the core question of the project: **which features actually influence price?**

### 5.1 Numerical vs. numerical

In [ ]:
#numerical vs numerical


**Findings for Area vs Price:**
- Scatter plot: no clear visual relationship
- Correlation (0.00154): almost no linear relationship
- Regression plot: nearly flat fitted line
- Mean price by area group: no consistent increase as area grows
- Standard deviation by group: large price variation inside every group

**Conclusion:** in this dataset `Area` alone is not a useful predictor of `Price`. That describes *this dataset only*; in real estate, area is normally very important.

In [ ]:
#area vs price
import matplotlib.pyplot as plt

plt.scatter(df['Area'], df['Price'])
plt.xlabel('Area')
plt.ylabel('Price')
plt.title('Area vs Price')
plt.show()

In [ ]:
df['Area'].corr(df['Price'])#correlation is nearly zero it mean weak linear relationship 

In [ ]:
df['Area'].describe()

In [ ]:
#we got nearly flat line
import seaborn as sns
import matplotlib.pyplot as plt

sns.regplot(data=df, x='Area', y='Price')
plt.show()

In [ ]:
df['AreaGroup'] = pd.qcut(df['Area'], q=4)
df.groupby('AreaGroup', observed=True)['Price'].mean()

In [ ]:
df.groupby('AreaGroup', observed=True)['Price'].std()

#### 📅 YearBuilt vs Price

In [ ]:
#yearbuilt vs price
#no pattern found
plt.scatter(df['YearBuilt'], df['Price'])
plt.xlabel('Year Built')
plt.ylabel('Price')
plt.title('Year Built vs Price')
plt.show()

In [ ]:
df['YearBuilt'].corr(df['Price'])#no co relation

### 5.2 Categorical vs. numerical
For each categorical feature we compare:
1. **Group means** → do average prices differ?
2. **Box plots** → do the price ranges differ?
3. **KDE plots** → do the price distributions overlap?

If a feature really mattered, we'd expect the groups to look different. Let's see.

#### 🛏️ Bedrooms vs Price

**Findings:** average prices are similar across bedroom counts, the box plots look alike, and group sizes are balanced, so the comparison is fair.

In [ ]:
#bedrooms vs price
#Bedroom count does not show a strong difference in average house prices in this dataset.

df.groupby('Bedrooms')['Price'].mean()

In [ ]:
#similar median and boxplot of bedroom categories
#Bedrooms does not appear to have a strong relationship with Price in  dataset,
sns.boxplot(data=df, x='Bedrooms', y='Price')
plt.title('Bedrooms vs Price')
plt.show()

In [ ]:
#group sizes are balanced 
df['Bedrooms'].value_counts().sort_index()

In [ ]:
sns.kdeplot(data=df, x='Price', hue='Bedrooms')
plt.title('Price Distribution by Number of Bedrooms')
plt.show()

#### 🛁 Bathrooms vs Price
Same pattern as bedrooms.

In [ ]:
#bathrooms vs price
#similar observations as above
df.groupby('Bathrooms')['Price'].mean()

In [ ]:
sns.boxplot(data=df, x='Bathrooms', y='Price')
plt.title('Bathrooms vs Price')
plt.show()

In [ ]:
df['Bathrooms'].value_counts().sort_index()

In [ ]:
#The bathroom categories have similar price distributions.highly overlap
sns.kdeplot(data=df, x='Price', hue='Bathrooms')
plt.title('Price Distribution by Number of Bathrooms')
plt.show()

#### 🏢 Floors vs Price
Same pattern again.

In [ ]:
##floors vs price -similar observations
df.groupby('Floors')['Price'].mean()

In [ ]:
sns.boxplot(data=df, x='Floors', y='Price')
plt.title('Floors vs Price')
plt.show()

In [ ]:
df['Floors'].value_counts().sort_index()

In [ ]:
sns.kdeplot(data=df, x='Price', hue='Floors')
plt.title('Price Distribution by Number of Floors')
plt.show()

#### 📍 Location vs Price
Surely location matters? (Spoiler: not here.)

In [ ]:
df.groupby('Location')['Price'].mean()

In [ ]:
sns.boxplot(data=df, x='Location', y='Price')
plt.title('Location vs Price')
plt.show()

In [ ]:
df['Location'].value_counts().sort_index()

In [ ]:
sns.kdeplot(data=df, x='Price', hue='Location')
plt.title('Price Distribution by Location')
plt.show()

#### 🚗 Garage vs Price

In [ ]:
df.groupby('Garage')['Price'].mean()

In [ ]:
sns.boxplot(data=df, x='Garage', y='Price')
plt.title('Garage vs Price')
plt.show()

In [ ]:
df['Garage'].value_counts().sort_index()

In [ ]:
sns.kdeplot(data=df, x='Price', hue='Garage')
plt.title('Price Distribution by Garage')
plt.show()

#### ⭐ Condition vs Price

In [ ]:
df.groupby('Condition')['Price'].mean()

In [ ]:
sns.boxplot(data=df, x='Condition', y='Price')
plt.title('Conditions vs Price')
plt.show()

In [ ]:
df['Condition'].value_counts().sort_index()

In [ ]:
sns.kdeplot(data=df, x='Price', hue='Condition')
plt.title('Price Distribution by Condition')
plt.show()

---
## 6️⃣ Correlation matrix 🔥
A heatmap that shows the linear correlation between every pair of numeric columns. Values near **+1 or -1** mean a strong relationship; values near **0** mean almost none.

> 🔎 **Main insight of this project:** look at the row for `Price`. Every cell is very close to zero.

In [ ]:
corr_matrix = df.select_dtypes(include='number').corr()
corr_matrix

In [ ]:
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

### Pair plot
A grid of scatter plots for every pair of numeric columns. No visible trends, just clouds of points. ☁️

In [ ]:
sns.pairplot(df)

---
## 7️⃣ Preparing data for the model 🔤
Machine learning models only understand **numbers**, so text columns must be encoded.

| Column | Type | Encoding used | Why |
|---|---|---|---|
| `Location` | Nominal (no order) | **One-Hot Encoding** | No location is "greater" than another |
| `Garage` | Binary (Yes/No) | **Map to 1/0** | Only two options |
| `Condition` | Ordinal (has order) | **Ordinal Encoding** (Poor=0 ... Excellent=3) | The order carries meaning |

We also remove the helper column `AreaGroup` that we created for analysis.

In [ ]:
df.head()

In [ ]:
df.drop(columns=['AreaGroup'], inplace=True)

We work on a **copy** (`df_encoded`) so the original data stays untouched.

In [ ]:
df_encoded = df.copy()

**One-hot encoding** `Location`. `drop_first=True` avoids the *dummy variable trap* (redundant columns that confuse linear models).

In [ ]:
#one hot encoding
location_encoded=pd.get_dummies(df['Location'], drop_first=True, dtype=int)
df_encoded = pd.concat(
    [df_encoded, location_encoded],
    axis=1
)


In [ ]:
df_encoded.drop('Location', axis=1, inplace=True)

**Binary encoding** for `Garage`.

In [ ]:
df_encoded['Garage'] = df_encoded['Garage'].map({
    'No': 0,
    'Yes': 1
})

In [ ]:
df_encoded['Garage'].value_counts()

**Ordinal encoding** for `Condition`.

In [ ]:
#ordinal encoding
df_encoded['Condition']=df['Condition'].map({
    'Poor': 0,
    'Fair': 1,
    'Good': 2,
    'Excellent': 3
})

In [ ]:
df_encoded['Condition'].value_counts()

---
## 8️⃣ Features, target & train-test split ✂️
- **X** = input features (everything except `Price` and `Id`)
- **y** = target (`Price`)

We keep **80%** of the data for training and **20%** for testing. The model never sees the test set during training, which gives an honest estimate of performance. `random_state=42` makes the split reproducible.

In [ ]:
X = df_encoded.drop(['Price', 'Id'], axis=1)
y = df_encoded['Price']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

---
## 9️⃣ Linear Regression model 🤖
Linear Regression learns a weight for each feature and combines them in a straight-line formula:

`Price = b0 + b1·Area + b2·Bedrooms + ...`

We **fit** it on the training data, then **predict** on the unseen test data.

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

lr.fit(X_train, y_train)

In [ ]:
y_pred = lr.predict(X_test)

---
## 🔟 Model evaluation 📏
We use **R² (coefficient of determination)**:
- **R² = 1** → perfect predictions
- **R² = 0** → no better than always guessing the average price
- **R² < 0** → worse than guessing the average

> The score printed below is the result of this project.

In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)

print("R2 Score:", r2)

### Extra metrics: how big are the errors in dollars?
R² alone is abstract. **MAE** and **RMSE** express the typical error in dollars, and a **baseline** (always predict the average price) shows what "no skill" looks like.

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# Baseline: always predict the average training price
baseline_pred = np.full(len(y_test), y_train.mean())
baseline_r2 = r2_score(y_test, baseline_pred)
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print(f"Model    -> R2: {r2:.4f} | MAE: ${mae:,.0f} | RMSE: ${rmse:,.0f}")
print(f"Baseline -> R2: {baseline_r2:.4f} | MAE: ${baseline_mae:,.0f}")

### Actual vs. predicted prices
If the model were good, the dots would hug the red diagonal line. If they form a flat horizontal band instead, the model is mostly predicting "the average" for everything.

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.4)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, 'r--', label='Perfect prediction')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted House Prices')
plt.legend()
plt.show()

---
## 1️⃣1️⃣ Trying a non-linear model: Random Forest 🌲
Linear Regression can only capture **straight-line** relationships. So a fair question is: *"Maybe the pattern exists, but it isn't linear?"*

To test this, I trained a **Random Forest Regressor**. It is a much more flexible model that can capture curves, interactions and thresholds. If hidden non-linear patterns existed, this model should find them.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

rf_r2 = r2_score(y_test, rf_pred)
print("Random Forest R2:", rf_r2)

### 📊 Model comparison
All three approaches on the same test set. The baseline simply predicts the average training price for every house.

In [ ]:
results = pd.DataFrame({
    'Model': ['Baseline (predict mean)', 'Linear Regression', 'Random Forest'],
    'R2': [baseline_r2, r2, rf_r2],
    'MAE ($)': [baseline_mae, mae, mean_absolute_error(y_test, rf_pred)]
})
results.round(4)

### 🔎 Interpreting the Random Forest result
The Random Forest scored an **R² of about -0.11**, which is **worse than the baseline**.

**Why does a more powerful model do worse?**
When features contain almost no real signal, a flexible model starts fitting *random noise* in the training data. That noise doesn't repeat in the test data, so predictions end up slightly worse than simply guessing the average. This is called **overfitting**.

**What this tells me:**
- Both a linear and a non-linear model fail, so the problem is **not the choice of algorithm**
- This supports the EDA findings: in this dataset, the features carry very little information about price
- A fancier model cannot create information that isn't in the data

> ⚠️ **Honest caveat:** I tested two models with default settings, so this is strong evidence, not proof. Hyperparameter tuning or feature engineering might change the numbers slightly, but with near-zero correlations I wouldn't expect a big jump.

---
# 🧠 Key Findings & Summary

## What I did
- Cleaned and inspected the dataset (no duplicates, no missing values)
- Performed **univariate** analysis on numerical and categorical features
- Performed **bivariate** analysis of every feature against `Price`
- Built a **correlation matrix** and pair plot
- Encoded categorical features (one-hot, binary and ordinal)
- Split the data 80/20 and trained a **Linear Regression** model
- Evaluated it with **R², MAE and RMSE**, compared against a mean-price baseline
- Tested a **Random Forest** to check whether a non-linear model could find hidden patterns

## What I found 🔎
| Observation | Evidence |
|---|---|
| Features are almost uncorrelated with Price | Area vs Price correlation ≈ **0.0015**; the heatmap row for Price is ~0 everywhere |
| Category groups look the same | Bedrooms, Bathrooms, Floors, Location, Garage and Condition have similar mean prices and heavily overlapping box/KDE plots |
| Distributions are flat and symmetric | Skewness ≈ 0 and no outliers for Area, YearBuilt and Price |
| Linear Regression R² ≈ -0.008 | Slightly below zero, so it is no better than predicting the average price |
| A non-linear model did not help | Random Forest scored R² ≈ **-0.11**, worse than the mean-price baseline (overfitting to noise) |

## 💡 Why did the model perform poorly?
It is **not** a coding mistake. The features simply don't contain much information about the target. Possible reasons:
1. The dataset's features have almost no real relationship with price. Flat distributions and near-zero correlations suggest it may be **synthetic (randomly generated)** data.
2. Real prices depend on things not in this dataset (exact neighbourhood, market conditions, school quality, etc.).
3. Linear Regression only captures linear patterns, but the Random Forest test showed that a non-linear model doesn't fix it either. Even tree-based models need *some* signal to find.

## 🎓 What I learned
- **Model quality = algorithm + data quality.** No algorithm can create information that isn't in the data.
- **EDA is not optional.** It told me what to expect before I trained anything.
- A more complex model is not automatically a better model. Random Forest did *worse* here because it overfit noise.
- A "bad" result is still a result. Understanding *why* it happened is the real learning.
- How to take a project from raw data to evaluation: cleaning → EDA → encoding → splitting → modelling → evaluating.

## 🚀 Next steps
- [ ] Try a richer, real-world housing dataset (e.g. Ames Housing or California Housing)
- [ ] Compare against Gradient Boosting (XGBoost / LightGBM)
- [ ] Add feature engineering (e.g. house age, price per sq. ft.)
- [ ] Use cross-validation for more reliable scores

---
*Thanks for reading! This is my first ML project, so feedback and suggestions are very welcome. 🙌*

---
## 💾 Saving plots for the README
This cell exports the key charts into an `images/` folder.

In [ ]:
import os
os.makedirs('images', exist_ok=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.savefig('images/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()

plt.figure(figsize=(8, 6))
sns.regplot(data=df, x='Area', y='Price', scatter_kws={'alpha': 0.4})
plt.title('Area vs Price (nearly flat line)')
plt.savefig('images/area_vs_price.png', dpi=150, bbox_inches='tight')
plt.close()

plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x='Bedrooms', y='Price')
plt.title('Bedrooms vs Price')
plt.savefig('images/bedrooms_vs_price.png', dpi=150, bbox_inches='tight')
plt.close()

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.4)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, 'r--', label='Perfect prediction')
plt.xlabel('Actual Price'); plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted House Prices')
plt.legend()
plt.savefig('images/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.close()